# PCB test-point placement — DreamerV3 cold-start training (A100, 20 traces)

Trains the placement policy **with the cold start** (expert demos + decayed behavior cloning + potential reward shaping) and optionally an **ablation** with all of it off (the previous setup), so you can compare the two directly in TensorBoard and against the classical baselines.

**Before running:** `Runtime → Change runtime type → A100 GPU` (Colab Pro).

Sized for an A100 at the canonical 20-trace board: demo generation ~10–15 min (one-time, cached), then a 100k-step run — roughly 3–6 h, with checkpoints every 5k steps, so you can interrupt/stop anytime and re-running the training cell **resumes automatically**. Enable the Drive cell so progress survives disconnects. On a free T4 instead, set `CONFIG = "colab"` and `NUM_TRACES = 8` in the settings cell (~2–4 h for 30k steps).

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > A100 GPU"
name = torch.cuda.get_device_name(0)
print("torch", torch.__version__, "|", name)
if "A100" not in name:
    print("NOTE: not an A100 — everything still runs, just slower; "
          "consider CONFIG='colab', NUM_TRACES=8 in the settings cell.")

In [ ]:
# Optional: persist logs + checkpoints in Google Drive (survives disconnects,
# and re-running the notebook later resumes training where it stopped).
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    LOGROOT = "/content/drive/MyDrive/pcb-router-logs"
else:
    LOGROOT = "/content/pcb-router/logdir"
print("logs ->", LOGROOT)

In [ ]:
# Get the code. Default: upload pcb-router-colab.zip (sits on your Desktop
# next to the repo). Alternative: push the branch and set USE_GITHUB = True.
import pathlib
USE_GITHUB = False
REPO_URL = "https://github.com/pauljiang03/pcb-router"
if not pathlib.Path("/content/pcb-router").exists():
    if USE_GITHUB:
        !git clone {REPO_URL} /content/pcb-router
    else:
        from google.colab import files
        print("Upload pcb-router-colab.zip ...")
        files.upload()
        !unzip -q -o pcb-router-colab.zip -d /content
%cd /content/pcb-router

In [ ]:
# Dependencies (torch/numpy/tensorboard/matplotlib ship with Colab) and a
# quick sanity run of the cold-start tests (~10 s).
%pip -q install gymnasium "ruamel.yaml" openpyxl
!python -m pytest tests/test_coldstart.py -q

In [ ]:
NUM_TRACES = 20        # canonical board size (T4 fallback: 8)
CONFIG = "colab_a100"  # configs.yaml section: steps/eval cadence/demos (T4 fallback: "colab")
# 4 env workers overlap CPU routing with GPU training (A100 VMs have 12 vCPUs).
# Set to "" if the worker processes misbehave.
ENV_FLAGS = "--envs 4 --parallel"
COLD_DIR = f"{LOGROOT}/cold"
ABLA_DIR = f"{LOGROOT}/ablation"
print(NUM_TRACES, "traces |", CONFIG, "|", COLD_DIR, "|", ABLA_DIR)

In [ ]:
# Live training curves. Key scalars: eval_return / train_return (totals are
# comparable across BOTH runs — shaping preserves episode totals), log_routed
# (nets routed, target = NUM_TRACES), log_phi (placement potential),
# bc_loss + bc_scale (imitation term; scale decays linearly to 0 over
# bc_decay steps — 40k on the A100 config).
%load_ext tensorboard
import tensorboard.notebook as tbnb
tbnb.start("--logdir " + LOGROOT)

## Run A — cold start (demos + BC + shaping)

First run generates 200 expert episodes into `demo_eps/` (progress prints every 10, ~10–15 min at 20 traces), then trains 100k steps with the `colab_a100` config. Interrupt anytime; re-run to resume.

In [ ]:
!python train.py --configs defaults {CONFIG} --logdir "{COLD_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS}

In [ ]:
# Score the trained policy against the classical baselines on the SAME boards
# (the summary table at the bottom is the headline result: compare the
# Dreamer row to Smart on failures / max / spread). Add --board_seed 1000000
# to score on held-out boards instead of the fixed TE board; add --fast for
# a quick low-budget pass (the Random baseline costs ~30s/episode at the
# quality budget on 20-trace boards).
!python eval.py --checkpoint "{COLD_DIR}/latest.pt" --configs defaults {CONFIG} \
    --episodes 3 --num_traces {NUM_TRACES} --device cuda:0 --no-plot

## Run B (optional) — ablation: the previous setup

Same budget, same boards, but no demos, no BC, terminal-only reward (`--demos 0 --bc_scale 0 --shaping none`). This is the apples-to-apples "before" run — compare `eval_return` curves in TensorBoard at equal steps.

In [ ]:
!python train.py --configs defaults {CONFIG} --logdir "{ABLA_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS} \
    --demos 0 --bc_scale 0 --shaping none

In [ ]:
!python eval.py --checkpoint "{ABLA_DIR}/latest.pt" --configs defaults {CONFIG} \
    --episodes 3 --num_traces {NUM_TRACES} --device cuda:0 --no-plot

## Reading the results

- **TensorBoard `eval_return`** — the main comparison. The cold-start run should reach demo-level return within the first few thousand steps (BC pulls the policy to the expert while the world model learns what routed placements look like); the ablation typically stays flat far below it — a near-constant return is the signature of the no-learning plateau (per-step validity bonuses + mediocre routing of effectively-random placements, with ~zero advantage signal reaching the actor).
- **`log_routed`** — routed nets per episode; the cold run should sit near `NUM_TRACES` early, the ablation well below.
- **`bc_scale`** — decays to 0 over `bc_decay` steps (40k here); improvement after that point is pure RL on top of the imitation floor.
- **`eval.py` summary tables** — the honest scoreboard vs. classical baselines on identical boards. Matching Smart on the TE board is success (it is near-optimal there); the learned policy's edge should show on held-out/moat boards (`--board_seed 1000000`) and in inference speed.

Even 100k steps is not a converged run — if the cold-start run clearly leads the ablation and is closing on Smart, the pipeline is working; scale steps up from there.

In [ ]:
# Package checkpoints + TensorBoard event files for download (replay buffer
# excluded to keep it small). Skip if you used Drive — it's already saved.
import pathlib, shutil
out = pathlib.Path("/content/results")
shutil.rmtree(out, ignore_errors=True)
for run in ("cold", "ablation"):
    d = pathlib.Path(LOGROOT) / run
    if d.exists():
        (out / run).mkdir(parents=True, exist_ok=True)
        for f in list(d.glob("events*")) + [d / "latest.pt"]:
            if f.exists():
                shutil.copy(f, out / run / f.name)
shutil.make_archive("/content/pcb_router_results", "zip", out)
from google.colab import files
files.download("/content/pcb_router_results.zip")